# RAG Framework — Architectural Specification

**Author:** Intern — 6-week applied RAG project  
**Reviewer:** Jesus  
**Implementation file:** `tragframe.py`  
**Purpose:** Design document and system specification. Not an experiment notebook.

> This notebook defines *what* the system does and *why* each decision was made.
> All runnable code lives in `tragframe.py`. Cells here are illustrative only.

---
# 1. Project Goal

## Problem

Given a local corpus of PDF documents (RAG papers, GCP guides, Git manuals), answer
natural-language questions using **only information that appears in those documents**.
The system must:

- never fabricate facts beyond what the documents contain
- cite which document chunk each claim comes from
- refuse when the context is insufficient

## Why local-first

| Constraint | Decision |
|------------|----------|
| No cloud budget | All computation runs on a single laptop |
| No API keys required | Embedding is deterministic; LLM via local Ollama |
| Reproducibility | Same input → same output every run, no external service calls |
| Review-readiness | Any component can be inspected, paused, or swapped without network access |

## Why deterministic baseline first

Before optimising for quality, the system must be **measurable**.
A deterministic baseline (fixed embedder, fixed chunking, fixed index) means:

- latency numbers are stable across runs
- retrieval metrics (Hit@k, Precision@k) reflect the system, not random noise
- any future change can be evaluated against a known starting point

Transformer embeddings are intentionally deferred until the baseline is fully measured.

---
# 2. High-Level RAG Pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│                         OFFLINE (build time)                        │
│                                                                     │
│   data/                                                             │
│   topic_A/*.pdf ──► ingest ──► chunk ──► embed ──► index           │
│   topic_B/*.pdf                                      │              │
│                                                      ▼              │
│                                               FAISS IndexFlatIP    │
└──────────────────────────────────────────────────────┬──────────────┘
                                                       │
┌──────────────────────────────────────────────────────▼──────────────┐
│                         ONLINE (query time)                         │
│                                                                     │
│   query ──► guardrails ──► embed ──► retrieve ──► quality gate      │
│                                                        │            │
│                                                   build prompt      │
│                                                        │            │
│                                                   LLM (Ollama)      │
│                                                        │            │
│                                                     answer          │
│                                                        │            │
│                                                    Monitor ──► artifacts/
└─────────────────────────────────────────────────────────────────────┘
```

## Stage descriptions

| Stage | What happens | Responsible component |
|-------|-------------|----------------------|
| **ingest** | Load PDFs page by page via PyPDFLoader; attach topic label from folder name | `VectorDatabase.update_database()` |
| **chunk** | Split pages into overlapping text windows | `RecursiveCharacterTextSplitter` inside `VectorDatabase` |
| **embed** | Convert each chunk text to a 768-dim float32 vector via HashingVectorizer | `VectorDatabase._embed()` |
| **index** | Store all vectors in FAISS IndexFlatIP; reset on each rebuild | `faiss.IndexFlatIP` inside `VectorDatabase` |
| **guardrails** | Reject queries containing PII, prompt injection, or toxic content | Module-level functions in `tragframe.py` |
| **retrieve** | Embed query; compute cosine similarity against index; return top-k Chunks | `VectorDatabase.recover()` |
| **quality gate** | Block if top-1 score < threshold or top-2 scores are too close | `retrieval_quality_gate()` |
| **build prompt** | Wrap chunks with citation labels and strict grounding rules | `RAG._build_prompt()` |
| **LLM** | Invoke local Ollama model; gracefully skip if unavailable | `RAG.retrieve()` |
| **report** | Write JSON artifacts for every stage of the run | `Monitor.write_json()` |

---
# 3. Framework Architecture

## UML — Class Relationships

```
┌───────────────────────────────┐
│          VectorDatabase       │
│───────────────────────────────│
│ + embedding_dim: int          │
│ + chunk_size: int             │
│ + chunk_overlap: int          │
│ + vectorizer: HashingVectorizer│
│ + index: faiss.IndexFlatIP    │
│ + chunks: List[Chunk]         │
│ + monitor: Monitor  ◄─────────┼──────────────────┐
│───────────────────────────────│                  │  shared (1-to-1)
│ + update_database(path)       │                  │
│ + recover(query, top_k)       │         ┌────────┴──────────┐
│ - _embed(texts)               │         │      Monitor      │
└───────────────┬───────────────┘         │───────────────────│
                │ owns 1                  │ + run_id: str     │
                │                         │ + timings: dict   │
                ▼                         │ + events: list    │
┌───────────────────────────────┐         │───────────────────│
│             RAG               │         │ + timeit(name)    │
│───────────────────────────────│         │ + log(event)      │
│ + vector_db: VectorDatabase   │         │ + write_json()    │
│ + monitor: Monitor  (shared)──┼────────►└───────────────────┘
│ + similarity_threshold: float │
│ + ambiguity_gap: float        │
│ + llm: OllamaLLM | None       │
│───────────────────────────────│
│ + retrieve(query, top_k)      │
│ - _build_prompt(hits, query)  │
└───────────────────────────────┘
```

**Key constraint:** One `RAG` instance holds exactly one `VectorDatabase`.
The `Monitor` is created inside `VectorDatabase` and shared by reference to `RAG`.
This means all timing and artifact data for a run flows through a single `Monitor` with a single `run_id`.

---

## 3.1 VectorDatabase

**Responsibility:** Own the entire offline pipeline (ingest → chunk → embed → index)
and the online retrieval step.

**Inputs:**
- `data_path` — root folder containing one subfolder per topic, each with `*.pdf` files
- `query` — plain text string at retrieval time

**Outputs:**
- `update_database()` → None (side-effects: populates `self.chunks` and `self.index`; writes artifacts)
- `recover(query, top_k)` → `List[RetrievalHit]`

**Design decisions:**
- `index.reset()` is called before every rebuild — the index is always rebuilt from scratch, never appended to incrementally. This eliminates stale-vector bugs.
- `max_pages_per_pdf` caps ingestion to prevent memory pressure from large PDFs.
- Topic label is derived from the subfolder name, not from document content. This is deterministic and requires no NLP.

---

## 3.2 RAG

**Responsibility:** Orchestrate the online query path: guardrails → retrieve → quality gate → prompt → LLM.

**Inputs:**
- A constructed `VectorDatabase` instance (passed in constructor)
- `query` — plain text string

**Outputs:**
- `retrieve(query, top_k)` → `str` (final answer, guardrail message, or refusal)

**Design decisions:**
- `RAG` never touches the FAISS index directly. All vector operations go through `VectorDatabase.recover()`.
- The LLM is optional. If Ollama is unavailable, `retrieve()` returns the retrieved context with a note — the pipeline does not crash.
- Guardrail functions are module-level (not methods), so they can be tested independently of any class instance.
- `similarity_threshold` and `ambiguity_gap` are constructor parameters — they are fixed per run, not computed dynamically.

---

## 3.3 Monitor

**Responsibility:** Record timing data and write JSON artifacts for every stage of a run.

**Key properties:**
- `run_id` — a 10-character UUID hex, generated once at construction. All artifact filenames are prefixed with this ID.
- `timings` — a flat dict updated by the `timeit()` context manager.

**Artifact naming convention:**
```
artifacts/
  {run_id}_00_config.json          ← VectorDatabase parameters
  {run_id}_10_ingest_summary.json  ← doc count, chunk count, errors
  {run_id}_20_chunks_sample.json   ← first 5 chunks (spot-check)
  {run_id}_30_retrieval_debug.json ← query, hits, scores, timings
  {run_id}_40_prompt_debug.json    ← prompt preview sent to LLM
  {run_id}_50_guardrails_report.json ← block reason if triggered
  {run_id}_60_generation_output.json ← LLM answer + model metadata
  {run_id}_90_timings.json         ← all stage timings for the run
```

The numeric prefix in filenames is intentional: it defines a human-readable execution order when sorted alphabetically.

---
# 4. Data Flow and Types

All data structures are defined as Python `dataclass` objects. No ORM, no Pydantic, no serialisation overhead.

## 4.1 Chunk

Produced by `VectorDatabase.update_database()`. Stored in `self.chunks`.

```python
@dataclass
class Chunk:
    text:   str   # raw chunk text, used for embedding and prompt context
    source: str   # PDF filename (e.g. "GitGuide.pdf")
    topic:  str   # folder name (e.g. "GIT", "RAG", "GCP")
    page:   int   # page number within the source PDF
```

**Note:** `Chunk` has no `chunk_id`. Position in `self.chunks` IS the ID — the FAISS index stores vectors at the same integer positions. `VectorDatabase.recover()` maps FAISS result indices directly back to `chunks[i]`.

## 4.2 RetrievalHit

Produced by `VectorDatabase.recover()`. Consumed by `RAG._build_prompt()`.

```python
@dataclass
class RetrievalHit:
    chunk: Chunk  # the full Chunk object
    score: float  # cosine similarity score (0.0 – 1.0 for normalised vectors)
    rank:  int    # 1-based rank in the result list
```

**Why embed `Chunk` inside `RetrievalHit` rather than just an index?**
The prompt builder needs `chunk.text`, `chunk.source`, `chunk.topic`, and `chunk.page` simultaneously.
Embedding the full object avoids a second lookup and makes `_build_prompt()` self-contained.

## 4.3 Evaluation Query Schema (Week 5)

Used in the evaluation notebook (`week5_evaluation_and_optimization.ipynb`). Not a dataclass — stored as a dict in `EVAL_QUERIES` and `EXPECTED_TOPIC`.

```python
# Implicit schema:
{
    'query':        str,          # natural language question
    'expected_kw':  str | None,   # topic keyword for heuristic auto-label
                                  # None = out-of-scope query
}
```

**`expected_kw` is a heuristic, not ground truth.** It checks whether the retrieved chunk's topic folder name contains the keyword. Manual labels in `label_df['manual_label']` override this for official metrics.

## 4.4 Data flow diagram

```
PDF file
   │
   ▼ PyPDFLoader
raw page text  +  {source, topic, page}
   │
   ▼ RecursiveCharacterTextSplitter
List[str]  (chunk texts)
   │
   ▼ Chunk(text, source, topic, page)
List[Chunk]  →  stored as self.chunks
   │
   ▼ HashingVectorizer + faiss.normalize_L2
np.ndarray  shape=(N, 768)  dtype=float32
   │
   ▼ faiss.IndexFlatIP.add()
FAISS index  (integer positions match self.chunks)
   │
   ▼  query time: IndexFlatIP.search(qv, top_k)
scores[], indices[]  →  List[RetrievalHit]
   │
   ▼  RAG._build_prompt()
prompt str  →  OllamaLLM.invoke()
   │
   ▼
answer str
```

---
# 5. Retrieval Strategy

## 5.1 Embedding: HashingVectorizer

`sklearn.feature_extraction.text.HashingVectorizer` maps each token to a fixed integer bucket
via a deterministic hash function (MurmurHash). The output is a sparse vector of token counts,
which is then converted to dense `float32` and L2-normalised.

**Why not a transformer (MiniLM, MPNet)?**

| Criterion | HashingVectorizer | SentenceTransformer |
|-----------|------------------|--------------------|
| Memory at runtime | < 50 MB | 400 MB – 1.5 GB |
| Kernel crash risk | None | OOM on large corpus |
| Cold-start time | Zero | Model download + load |
| Deterministic output | Yes (hash is fixed) | Yes (but GPU-dependent precision) |
| Semantic similarity | No (vocabulary only) | Yes |
| Suitable for baseline | **Yes** | Later upgrade |

HashingVectorizer was chosen because **no single crash is acceptable in a live demo**.
Semantic quality is a secondary concern for the baseline measurement.

## 5.2 Similarity: cosine via L2 normalisation

All vectors (corpus and query) are L2-normalised before entering FAISS:

```python
faiss.normalize_L2(x)   # in-place, modifies x
```

After normalisation, inner product equals cosine similarity:

```
cos(a, b) = (a · b) / (‖a‖ · ‖b‖)  =  a · b   (when ‖a‖ = ‖b‖ = 1)
```

Using `IndexFlatIP` (inner product) on normalised vectors gives exact cosine ranking
without a separate normalisation index type.

## 5.3 Index: FAISS IndexFlatIP

- **Exact search** — every vector is compared against every query vector
- **No training required** — vectors are added directly
- **Deterministic results** — same query always returns the same ranked list
- **Trade-off:** O(N · d) per query; scales linearly with corpus size

For the current corpus (hundreds to low thousands of chunks), exact search is fast enough.
Approximate search (HNSW) is evaluated in Week 5 as a comparison — it is not the operational index.

## 5.4 Retrieval quality gate

After retrieval, two conditions are checked before the LLM is called:

```
top1_score < similarity_threshold   →  NO_CONTEXT
(scores[0] - scores[1]) < ambiguity_gap  →  AMBIGUOUS_RETRIEVAL
```

Both thresholds are constructor parameters of `RAG`, not hard-coded constants.
This makes them testable and auditable per run.

## 5.5 No LLM in baseline retrieval

The retrieval path (`VectorDatabase.recover()`) is completely independent of the LLM.
Retrieval quality can be measured (Hit@k, Precision@k) without ever invoking Ollama.
This separation is intentional: retrieval metrics must be reproducible even when Ollama is offline.

---
# 6. Evaluation Design

Evaluation is implemented in `notebooks/week5_evaluation_and_optimization.ipynb`.
It wraps the existing pipeline with measurement logic — it does **not** modify `tragframe.py`.

## 6.1 Metrics defined

### Hit@k

For a given query, at least one of the top-k retrieved chunks is relevant.

```
Hit@k(q) = 1  if  any(relevant(chunk_i))  for i in top-k results
Hit@k(q) = 0  otherwise

Macro-avg Hit@k = mean( Hit@k(q) )  over all queries
```

### Precision@k

Fraction of the top-k retrieved chunks that are relevant.

```
Prec@k(q) = count(relevant chunks in top-k) / k

Macro-avg Prec@k = mean( Prec@k(q) )  over all queries
```

## 6.2 Relevance labels

Two label types are maintained in `label_df`:

| Column | Source | Use |
|--------|--------|-----|
| `auto_label` | Topic keyword match (heuristic) | Estimated metrics only — not authoritative |
| `manual_label` | Human judgment on `text_preview` | **Official metrics** — authoritative |

Official metrics are computed exclusively on rows where `manual_label` is set.
Coverage is reported: if only 6 of 10 queries are labeled, the official metric
covers 6 queries and this is documented explicitly.

## 6.3 When metrics are computed

```
VectorDatabase.recover()  →  List[RetrievalHit]
                                      │
                              [evaluation layer]
                            label_df + manual_label
                                      │
                              Hit@k, Precision@k
                                      │
                            pandas DataFrame → display
```

Metrics are computed post-retrieval on the `RetrievalHit` list — they are never
inside `tragframe.py` and do not affect the retrieval path.

## 6.4 Why evaluation is outside the framework class

Evaluation requires human labels, which are not available at framework construction time.
Embedding evaluation logic inside `RAG` or `VectorDatabase` would couple the production
retrieval path to a measurement concern. Keeping it external means:

- `tragframe.py` can be used in production without any evaluation overhead
- evaluation can be re-run with different label sets without touching the framework
- evaluation notebooks are independently reviewable

---
# 7. Determinism and Reproducibility

Every design decision that affects output stability is documented here.

## 7.1 Embedding is deterministic by design

`HashingVectorizer` uses MurmurHash with a fixed seed. The same token always maps
to the same hash bucket. There is no random initialisation, no dropout, no sampling.

```python
HashingVectorizer(
    n_features=768,         # fixed dimension — never changes mid-run
    alternate_sign=False,   # positive-only weights — simpler dot-product
    norm='l2',              # L2 normalised at vectorizer level
)
```

## 7.2 Chunking is deterministic

`RecursiveCharacterTextSplitter` is a rule-based splitter with fixed parameters:

```python
chunk_size    = 300   # characters
chunk_overlap = 50    # characters
separators    = ['\n\n', '\n', '. ', ' ', '']
```

Given the same PDF and the same parameters, the same chunks are produced every time.
The separator priority list is fixed and not configurable at runtime.

## 7.3 Index is rebuilt from scratch on every `update_database()` call

```python
self.index.reset()   # clears all stored vectors
self.index.add(vecs) # adds fresh vectors
self.chunks = new_chunks
```

This means `chunks[i]` always corresponds to the vector at position `i` in the index.
There are no stale or orphaned vectors.

## 7.4 Run artifact logging

Every run produces a unique `run_id` (UUID hex, 10 chars). Artifacts are namespaced by this ID:

```
artifacts/3f9a1c72b8_00_config.json
artifacts/3f9a1c72b8_90_timings.json
```

This means:
- Multiple runs never overwrite each other
- Any past run can be audited by its `run_id`
- Config and timings are always co-located in the same artifact directory

## 7.5 LLM temperature

`temperature=0.0` is the default for evaluation runs. This makes LLM outputs
deterministic (greedy decoding) for the purposes of prompt comparison scoring.

---
# 8. Run Modes

## 8.1 Interactive mode (demo / development)

Used during development and mentor reviews. Single query, immediate output.

```python
# From tragframe.py __main__ block:
vd = VectorDatabase(embedding_dim=768, chunk_size=300,
                    chunk_overlap=50, max_pages_per_pdf=20)
vd.update_database("data")

rag = RAG(vd, model="gemma3:4b", temperature=0.2,
          similarity_threshold=0.30, ambiguity_gap=0.05)

answer = rag.retrieve("What is RAG?", top_k=3)
print(answer)
```

Run with: `python tragframe.py`

## 8.2 Batch evaluation mode (Week 5)

Used in `week5_evaluation_and_optimization.ipynb`. Runs all `EVAL_QUERIES` through
the retrieval pipeline, collects results into a pandas DataFrame, and computes metrics.

The framework is imported as-is — no code in `tragframe.py` is modified.

## 8.3 Artifact directory structure

```
artifacts/
│
├── {run_id_A}_00_config.json
├── {run_id_A}_10_ingest_summary.json
├── {run_id_A}_20_chunks_sample.json
├── {run_id_A}_30_retrieval_debug.json
├── {run_id_A}_40_prompt_debug.json
├── {run_id_A}_50_guardrails_report.json   ← only if guardrail triggered
├── {run_id_A}_60_generation_output.json
├── {run_id_A}_90_timings.json
│
├── {run_id_B}_00_config.json              ← next run, separate namespace
└── ...
```

Artifacts grow indefinitely. No automatic cleanup — all past runs are retained for audit.

---
# 9. Scope Boundaries

These constraints are intentional and non-negotiable for this project phase.

| Excluded | Reason |
|----------|--------|
| Cloud storage / APIs | No external dependencies; reproducible on any laptop |
| LangChain abstractions | Used only for PDF loading and text splitting (utilities, not architecture) — the RAG logic itself does not depend on LangChain |
| Agent loop / tool use | Out of scope; adds non-determinism and complexity with no current benefit |
| Vector DB services (Pinecone, Weaviate) | Adds network dependency; FAISS covers current corpus size |
| Fine-tuning | No training infrastructure available; evaluation first |
| Streaming responses | Complicates evaluation; deferred |
| Multi-tenancy | Single-user local system |
| Persistent index (disk) | Index is rebuilt on startup; corpus is small enough |

**On LangChain:** `PyPDFLoader` and `RecursiveCharacterTextSplitter` are used as utility
functions — they load files and split strings. They are not part of the architectural design.
Replacing them would require two function swaps and zero structural change.

**On overengineering:** The framework has three classes and four module-level functions.
Every line in `tragframe.py` can be read and explained in under two minutes.
Complexity is added only when a new requirement is demonstrated, not anticipated.

---
# 10. Mapping Weeks to Framework

Each week of the internship corresponds to one layer of `tragframe.py`.
The framework was designed so that each week's deliverable slots into the same skeleton
without requiring structural changes to prior weeks.

```
tragframe.py
│
├── Chunk, RetrievalHit               ← data types shared across all weeks
│
├── HashingVectorizer + FAISS         ← Week 1 + Week 2
│   _embed(), update_database(),      │
│   recover()                         │
│                                     │
├── RAG._build_prompt()               ← Week 3
│   OllamaLLM integration             │
│                                     │
├── check_pii()                       ← Week 4
│   check_prompt_injection()          │
│   check_toxicity()                  │
│   retrieval_quality_gate()          │
│                                     │
└── Monitor (run_id, timeit,          ← Week 5
    write_json, artifacts/)           │
    + evaluation notebook             │
```

## Week-by-week detail

| Week | Focus | Framework contribution |
|------|-------|------------------------|
| **Week 1** | Embedding theory: what are vector representations, cosine similarity, MiniLM vs MPNet | Establishes why `_embed()` uses L2 normalisation; justifies the dot-product index |
| **Week 2** | Ingestion pipeline: PDF loading, chunking strategy, FAISS index construction | Implements `VectorDatabase.update_database()` and `recover()`; produces the operational baseline |
| **Week 3** | Generation: prompt design, LLM integration, context formatting | Implements `RAG._build_prompt()` and `RAG.retrieve()`; introduces `OllamaLLM` with graceful fallback |
| **Week 4** | Guardrails: input validation, retrieval quality control, refusal logic | Adds `check_pii()`, `check_prompt_injection()`, `check_toxicity()`, `retrieval_quality_gate()` |
| **Week 5** | Evaluation: latency measurement, retrieval metrics, prompt comparison, vector store comparison | Adds `Monitor` with artifact logging; wraps pipeline in evaluation notebook without modifying `tragframe.py` |

## Design principle: additive, not invasive

Each week adds capability without modifying what existed before:

- Week 3 does not change `VectorDatabase` — it only adds `RAG` on top
- Week 4 does not change retrieval — it only adds pre/post checks
- Week 5 does not change `RAG` or `VectorDatabase` — it only adds a measurement wrapper

This is the core architectural discipline: **extension over modification**.

---

## Implementation reference

All classes, functions, and constants described above are implemented in `tragframe.py`.
The entry point for a demo run is the `__main__` block at the bottom of that file.

```
python tragframe.py
```

Evaluation is in:
```
notebooks/week5_evaluation_and_optimization.ipynb
```

This specification notebook contains no executable production code and requires no dependencies to open.